# 02 - Enriquecimiento y unificacion

Objetivo: leer `RAW.TRIPS`, enriquecer con Taxi Zones y normalizar catalogos para dejar una tabla unificada lista para la OBT.

In [1]:
import os
import requests
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [2]:
spark_snowflake_pkg = os.getenv('SPARK_SNOWFLAKE_PACKAGE', 'net.snowflake:spark-snowflake_2.12:3.1.8')
snowflake_jdbc_pkg = os.getenv('SNOWFLAKE_JDBC_PACKAGE', 'net.snowflake:snowflake-jdbc:3.14.4')

spark = (
    SparkSession.builder
    .appName('PSet3 Enriquecimiento Unificacion')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.jars.packages', f"{spark_snowflake_pkg},{snowflake_jdbc_pkg}")
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)

print(spark.version)

3.5.0


## 1) Configuracion

In [3]:
def get_env(name, default=None, required=False):
    value = os.getenv(name, default)
    if required and (value is None or str(value).strip() == ''):
        raise ValueError(f'Falta variable de entorno requerida: {name}')
    return value

snowflake_host = get_env('SNOWFLAKE_HOST', '').strip()
snowflake_account = get_env('SNOWFLAKE_ACCOUNT', '').strip()
snowflake_port = get_env('SNOWFLAKE_PORT', '443').strip()

if not snowflake_host:
    if not snowflake_account:
        raise ValueError('Debes configurar SNOWFLAKE_HOST o SNOWFLAKE_ACCOUNT')
    snowflake_host = f"{snowflake_account}.snowflakecomputing.com"

if snowflake_port and snowflake_port != '443':
    snowflake_host = f"{snowflake_host}:{snowflake_port}"

sf_options = {
    'sfURL': snowflake_host,
    'sfUser': get_env('SNOWFLAKE_USER', required=True),
    'sfPassword': get_env('SNOWFLAKE_PASSWORD', required=True),
    'sfRole': get_env('SNOWFLAKE_ROLE', required=True),
    'sfWarehouse': get_env('SNOWFLAKE_WAREHOUSE', required=True),
    'sfDatabase': get_env('SNOWFLAKE_DATABASE', required=True),
    'sfSchema': get_env('SNOWFLAKE_SCHEMA_RAW', 'RAW'),
}

raw_schema = get_env('SNOWFLAKE_SCHEMA_RAW', 'RAW')
raw_table = f"{raw_schema}.TRIPS"
unified_table = f"{raw_schema}.TRIPS_UNIFIED"
taxi_zone_lookup_url = get_env('TAXI_ZONE_LOOKUP_URL', required=True)

{k: ('***' if 'Password' in k else v) for k, v in sf_options.items()}

{'sfURL': 'DVNRQOK-CRC15844.snowflakecomputing.com',
 'sfUser': 'tonyfajardo1d',
 'sfPassword': '***',
 'sfRole': 'ACCOUNTADMIN',
 'sfWarehouse': 'PSET3_WH',
 'sfDatabase': 'DM_PSET3',
 'sfSchema': 'RAW'}

## 2) Leer RAW.TRIPS desde Snowflake

In [4]:
df_raw = (
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('dbtable', raw_table)
    .load()
)

print('rows raw =', df_raw.count())
df_raw.printSchema()

rows raw = 866674985
root
 |-- TRIP_NK: string (nullable = true)
 |-- PICKUP_DATETIME: timestamp (nullable = true)
 |-- DROPOFF_DATETIME: timestamp (nullable = true)
 |-- PU_LOCATION_ID: decimal(38,0) (nullable = true)
 |-- DO_LOCATION_ID: decimal(38,0) (nullable = true)
 |-- PASSENGER_COUNT: double (nullable = true)
 |-- TRIP_DISTANCE: double (nullable = true)
 |-- RATE_CODE_ID: decimal(38,0) (nullable = true)
 |-- PAYMENT_TYPE: decimal(38,0) (nullable = true)
 |-- VENDOR_ID: decimal(38,0) (nullable = true)
 |-- TRIP_TYPE: decimal(38,0) (nullable = true)
 |-- STORE_AND_FWD_FLAG: string (nullable = true)
 |-- FARE_AMOUNT: double (nullable = true)
 |-- EXTRA: double (nullable = true)
 |-- MTA_TAX: double (nullable = true)
 |-- TIP_AMOUNT: double (nullable = true)
 |-- TOLLS_AMOUNT: double (nullable = true)
 |-- IMPROVEMENT_SURCHARGE: double (nullable = true)
 |-- CONGESTION_SURCHARGE: double (nullable = true)
 |-- AIRPORT_FEE: double (nullable = true)
 |-- TOTAL_AMOUNT: double (nullable

## 3) Cargar Taxi Zones y enriquecer PU/DO

In [5]:
data_dir = Path('/home/jovyan/work/data')
data_dir.mkdir(parents=True, exist_ok=True)
zones_path = data_dir / 'taxi_zone_lookup.csv'

if not zones_path.exists():
    r = requests.get(taxi_zone_lookup_url, timeout=120)
    r.raise_for_status()
    zones_path.write_bytes(r.content)

df_zones = spark.read.csv(str(zones_path), header=True, inferSchema=True)
df_zones.show(5, truncate=False)

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows



In [6]:
df_pu = df_zones.select(
    F.col('LocationID').cast('int').alias('pu_location_id'),
    F.col('Borough').alias('pu_borough'),
    F.col('Zone').alias('pu_zone')
)

df_do = df_zones.select(
    F.col('LocationID').cast('int').alias('do_location_id'),
    F.col('Borough').alias('do_borough'),
    F.col('Zone').alias('do_zone')
)

df_enriched_zones = (
    df_raw
    .join(df_pu, on='pu_location_id', how='left')
    .join(df_do, on='do_location_id', how='left')
)

df_enriched_zones.select('service_type', 'pu_location_id', 'pu_zone', 'do_location_id', 'do_zone').show(10, truncate=False)

+------------+--------------+-----------------------------+--------------+---------------------+
|service_type|pu_location_id|pu_zone                      |do_location_id|do_zone              |
+------------+--------------+-----------------------------+--------------+---------------------+
|yellow      |140           |Lenox Hill East              |161           |Midtown Center       |
|yellow      |229           |Sutton Place/Turtle Bay North|236           |Upper East Side North|
|yellow      |236           |Upper East Side North        |239           |Upper West Side South|
|yellow      |43            |Central Park                 |162           |Midtown East         |
|yellow      |249           |West Village                 |90            |Flatiron             |
|yellow      |140           |Lenox Hill East              |140           |Lenox Hill East      |
|yellow      |24            |Bloomingdale                 |163           |Midtown North        |
|yellow      |143           |L

## 4) Mapear catalogos (vendor, rate code, payment, trip type)

In [7]:
vendor_map = F.create_map(
    F.lit(1), F.lit('Creative Mobile Technologies'),
    F.lit(2), F.lit('VeriFone Inc.')
)

rate_code_map = F.create_map(
    F.lit(1), F.lit('Standard rate'),
    F.lit(2), F.lit('JFK'),
    F.lit(3), F.lit('Newark'),
    F.lit(4), F.lit('Nassau or Westchester'),
    F.lit(5), F.lit('Negotiated fare'),
    F.lit(6), F.lit('Group ride')
)

payment_map = F.create_map(
    F.lit(1), F.lit('Credit card'),
    F.lit(2), F.lit('Cash'),
    F.lit(3), F.lit('No charge'),
    F.lit(4), F.lit('Dispute'),
    F.lit(5), F.lit('Unknown'),
    F.lit(6), F.lit('Voided trip')
)

trip_type_map = F.create_map(
    F.lit(1), F.lit('Street-hail'),
    F.lit(2), F.lit('Dispatch')
)

df_unified = (
    df_enriched_zones
    .withColumn('vendor_name', vendor_map[F.col('vendor_id')])
    .withColumn('rate_code_desc', rate_code_map[F.col('rate_code_id')])
    .withColumn('payment_type_desc', payment_map[F.col('payment_type')])
    .withColumn('trip_type_desc', trip_type_map[F.col('trip_type')])
    .withColumn('vendor_name', F.coalesce(F.col('vendor_name'), F.lit('Unknown')))
    .withColumn('rate_code_desc', F.coalesce(F.col('rate_code_desc'), F.lit('Unknown')))
    .withColumn('payment_type_desc', F.coalesce(F.col('payment_type_desc'), F.lit('Unknown')))
    .withColumn('trip_type_desc', F.coalesce(F.col('trip_type_desc'), F.lit('Unknown')))
)

df_unified.select('service_type', 'vendor_id', 'vendor_name', 'rate_code_id', 'rate_code_desc', 'payment_type', 'payment_type_desc', 'trip_type', 'trip_type_desc').show(20, truncate=False)

+------------+---------+----------------------------+------------+--------------+------------+-----------------+---------+--------------+
|service_type|vendor_id|vendor_name                 |rate_code_id|rate_code_desc|payment_type|payment_type_desc|trip_type|trip_type_desc|
+------------+---------+----------------------------+------------+--------------+------------+-----------------+---------+--------------+
|yellow      |2        |VeriFone Inc.               |1           |Standard rate |1           |Credit card      |NULL     |Unknown       |
|yellow      |1        |Creative Mobile Technologies|1           |Standard rate |2           |Cash             |NULL     |Unknown       |
|yellow      |2        |VeriFone Inc.               |1           |Standard rate |1           |Credit card      |NULL     |Unknown       |
|yellow      |2        |VeriFone Inc.               |1           |Standard rate |1           |Credit card      |NULL     |Unknown       |
|yellow      |2        |VeriFone I

## 5) Escritura de tabla unificada

In [8]:
(
    df_unified.write
    .format('snowflake')
    .options(**sf_options)
    .option('dbtable', unified_table)
    .mode('overwrite')
    .save()
)

print(f'Escritura OK -> {unified_table}')

Escritura OK -> RAW.TRIPS_UNIFIED


## 6) Verificaciones rapidas

In [9]:
q_counts = f"""
SELECT service_type, source_year, source_month, COUNT(*) AS rows_unified
FROM {unified_table}
GROUP BY 1,2,3
ORDER BY 1,2,3
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_counts)
    .load()
    .show(40, truncate=False)
)

+------------+-----------+------------+------------+
|SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|ROWS_UNIFIED|
+------------+-----------+------------+------------+
|green       |2015       |01          |1506268     |
|green       |2015       |02          |1572532     |
|green       |2015       |03          |1720117     |
|green       |2015       |04          |1661968     |
|green       |2015       |05          |1784221     |
|green       |2015       |06          |1636378     |
|green       |2015       |07          |1539148     |
|green       |2015       |08          |1529833     |
|green       |2015       |09          |1492510     |
|green       |2015       |10          |1627603     |
|green       |2015       |11          |1526922     |
|green       |2015       |12          |1605152     |
|green       |2016       |01          |1442315     |
|green       |2016       |02          |1507812     |
|green       |2016       |03          |1573184     |
|green       |2016       |04          |1540967

In [10]:
q_nulls = f"""
SELECT
  SUM(CASE WHEN pu_zone IS NULL THEN 1 ELSE 0 END) AS null_pu_zone,
  SUM(CASE WHEN do_zone IS NULL THEN 1 ELSE 0 END) AS null_do_zone,
  SUM(CASE WHEN payment_type_desc = 'Unknown' THEN 1 ELSE 0 END) AS unknown_payment_desc
FROM {unified_table}
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_nulls)
    .load()
    .show(truncate=False)
)

+------------+------------+--------------------+
|NULL_PU_ZONE|NULL_DO_ZONE|UNKNOWN_PAYMENT_DESC|
+------------+------------+--------------------+
|0           |0           |22831327            |
+------------+------------+--------------------+

